# Paper B — Cross-dataset generalisation of 12-lead ECG classification (PTB-XL ⇄ Chapman-Shaoxing)

Runs top-to-bottom on Colab GPU. Inputs = outputs of `01_convert_to_numpy.ipynb`.
Outputs (models, predictions, figures, tables, `results.json`) go to `OUT_DIR` on Google Drive.
**Resumable**: finished runs are skipped, unfinished runs continue from their last epoch checkpoint.

Protocol (fixed before seeing results):
- Forward (main): train PTB-XL folds 1–8, select epoch + thresholds on fold 9, test fold 10 (in-domain)
  and **all** Chapman (external, never used for training/tuning).
- Reverse (secondary): train Chapman 80 % / val 10 % / test 10 % (1 ECG per patient ⇒ record split = patient split),
  external = all PTB-XL.
- Multi-label, sigmoid + class-weighted BCE. Metrics: macro/per-class AUROC, macro-F1 with val-chosen thresholds,
  per-class sensitivity/specificity; 3 seeds (mean ± std) + bootstrap 95 % CIs.

In [ ]:
# ======================= CONFIG =======================
import os
FAST_MODE = os.environ.get('FAST_MODE', '1') == '1'   # True: tiny smoke test (minutes). False: the real experiment.

DATA_DIR = os.environ.get('CONV_OUT', '/content/drive/MyDrive/Research/paperB_data')      # converter output
OUT_DIR  = os.environ.get('PAPERB_OUT', '/content/drive/MyDrive/Research/paperB_outputs')
LOCAL_CACHE = '/content/cache'          # data copied here from Drive for fast access (Colab only)

FS, REC_LEN, WIN = 100, 1000, 250       # 100 Hz, 10 s records, 2.5 s windows (Strodthoff et al. 2021)
TEST_STRIDE = WIN // 2                  # test-time windows overlap by half, aggregated by element-wise max
BANDPASS = (0.5, 40.0)                  # identical zero-phase Butterworth band-pass for BOTH datasets
FILTER_ORDER = 3

MODELS         = ['xresnet1d101', 'inception1d', 'cnn1d', 'cnn_bilstm']
REVERSE_MODELS = ['xresnet1d101', 'inception1d']     # Chapman -> PTB-XL direction ([] to skip)
ABLATION_NOFILTER_MODEL = 'xresnet1d101'             # H4: same model without band-pass (None to skip)
SEEDS = [0, 1, 2]

EPOCHS, PATIENCE, BATCH, LR, WD = 50, 10, 128, 3e-3, 1e-2
POS_WEIGHT_CLIP = 100.0
N_BOOT = 1000
# label selection: a concept is used only if it has enough positives in BOTH datasets
MIN_POS = {'ptb_train': 100, 'ptb_test': 10, 'chapman': 50}
PTB_MIN_LIKELIHOOD = 0     # PTB-XL rhythm statements all carry likelihood 0, so 0 keeps everything (as in benchmark)
INCLUDE_TIER_B = True      # Tier B = loosely matched label definitions (analysed separately)

if FAST_MODE:
    MODELS, REVERSE_MODELS, SEEDS = ['xresnet1d101', 'inception1d', 'cnn1d', 'cnn_bilstm'], ['cnn1d'], [0, 1]
    EPOCHS, PATIENCE, N_BOOT, BATCH = 3, 3, 50, 64
    MIN_POS = {'ptb_train': 15, 'ptb_test': 2, 'chapman': 8}
    FAST_MAX_TRAIN, FAST_MAX_EXT = 3000, 3000

In [ ]:
import sys, json, time, ast, shutil, math, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and (OUT_DIR.startswith('/content/drive') or DATA_DIR.startswith('/content/drive')):
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('!!! Drive mount failed (VS Code extension?):', e)
        print('!!! Outputs go to local /content/paperB_outputs and will NOT survive a disconnect.')
        OUT_DIR = '/content/paperB_outputs'

import torch, torch.nn as nn, torch.nn.functional as F
from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib
matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12, 'figure.dpi': 110})
warnings.filterwarnings('ignore', category=UserWarning)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path(OUT_DIR)
for sub in ['checkpoints', 'preds', 'figures', 'tables']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
FIG, TAB = OUT / 'figures', OUT / 'tables'
print('device:', DEV, torch.cuda.get_device_name(0) if DEV.type == 'cuda' else '', '| FAST_MODE:', FAST_MODE)
print('outputs ->', OUT)


def savefig(fig, name):
    fig.tight_layout(); fig.savefig(FIG / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show() if IN_COLAB else None; plt.close(fig)

## 1. Load converted data

In [ ]:
src = Path(DATA_DIR)
if IN_COLAB and str(src).startswith('/content/drive'):
    dst = Path(LOCAL_CACHE); dst.mkdir(parents=True, exist_ok=True)
    for f in ['ptbxl_X.npy', 'chapman_X.npy', 'ptbxl_meta.csv', 'chapman_meta.csv', 'conversion_report.json']:
        if not (dst / f).exists() or (dst / f).stat().st_size != (src / f).stat().st_size:
            print('copying', f, 'to local disk ...'); shutil.copy(src / f, dst / f)
    src = dst
conv_report = json.load(open(src / 'conversion_report.json'))
SYNTHETIC = bool(conv_report.get('SYNTHETIC', False))
BANNER = ('#' * 78 + '\n#  SYNTHETIC TEST DATA — every number below is MEANINGLESS. Do not report.  #\n' + '#' * 78) \
    if SYNTHETIC else ('=' * 78 + '\n   REAL DATA (PTB-XL + Chapman). Numbers are real results.\n' + '=' * 78)
print(BANNER)
if 'UNIT_WARNING' in conv_report:
    raise RuntimeError('Converter flagged a unit problem: ' + conv_report['UNIT_WARNING'])

ptb_meta = pd.read_csv(src / 'ptbxl_meta.csv')
ch_meta = pd.read_csv(src / 'chapman_meta.csv', dtype={'snomed': str, 'beat': str, 'rhythm': str})
X_ptb_raw = np.load(src / 'ptbxl_X.npy', mmap_mode='r')
X_ch_raw = np.load(src / 'chapman_X.npy', mmap_mode='r')
CH_FORMAT = conv_report['chapman_format']
print('PTB-XL:', X_ptb_raw.shape, '| Chapman:', X_ch_raw.shape, f'({CH_FORMAT})')
print(json.dumps({k: conv_report[k] for k in ['ptbxl', 'chapman', 'p2p_ratio_chapman_over_ptbxl']}, indent=1))

## 2. Label harmonisation
Common concepts named after the SNOMED-CT scored classes of the PhysioNet/CinC Challenge 2021
(Reyna et al. 2022, Table 5). Pairs the Challenge scored as identical are merged
(CRBBB+RBBB, CLBBB+LBBB, PAC+SVPB, PVC+VPB).

- **Tier A**: definitions match closely (rhythm, conduction blocks, ectopy).
- **Tier B**: names match but criteria plausibly differ between Schiller (PTB-XL) and GE MUSE (Chapman) reports
  (hypertrophy, voltage, T/Q-wave, axis). Analysed separately — this is hypothesis H3.

⚠ Chapman acronyms: Rhythm codes are from Zheng et al. 2020 Table 2 (note **Chapman `AF` = atrial FLUTTER**,
`AFIB` = fibrillation). Beat/condition acronyms must be checked against `AttributesDictionary.xlsx`;
every acronym that is NOT mapped is printed below with its count so nothing is silently dropped.

In [ ]:
# concept: (tier, snomed codes [Reyna 2022 Table 5 where available], PTB-XL scp codes, Chapman acronyms)
CONCEPTS = {
    'AF':    ('A', ['164889003'], {'AFIB'}, {'AFIB'}),
    'AFL':   ('A', ['164890007'], {'AFLT'}, {'AF', 'AFL'}),
    'SR':    ('A', ['426783006'], {'SR'}, {'SR'}),
    'SB':    ('A', ['426177001'], {'SBRAD'}, {'SB'}),
    'STach': ('A', ['427084000'], {'STACH'}, {'ST'}),
    'SA':    ('A', ['427393009'], {'SARRH'}, {'SI', 'SA'}),
    'SVT':   ('A', [], {'SVTAC', 'PSVT'}, {'SVT', 'AT', 'AVNRT', 'AVRT', 'SAAWR'}),
    'IAVB':  ('A', ['270492004'], {'1AVB'}, {'1AVB'}),
    'RBBB':  ('A', ['59118001', '713427006'], {'CRBBB'}, {'RBBB', 'CRBBB'}),
    'IRBBB': ('A', ['713426002'], {'IRBBB'}, {'IRBBB'}),
    'LBBB':  ('A', ['164909002', '733534002'], {'CLBBB'}, {'LBBB', 'CLBBB'}),
    'LAnFB': ('A', ['445118002'], {'LAFB'}, {'LAFB', 'LAnFB', 'LFBBB'}),
    'PVC':   ('A', ['427172004', '17338001'], {'PVC'}, {'VPB', 'VEB', 'PVC', 'VPE'}),
    'PAC':   ('A', ['284470004', '63593006'], {'PAC'}, {'APB', 'PAC', 'SVPB'}),
    'WPW':   ('A', [], {'WPW'}, {'WPW'}),
    'LVH':   ('B', [], {'LVH'}, {'LVH'}),
    'LQRSV': ('B', ['251146004'], {'LVOLT'}, {'LVQRSAL', 'LVQRSCL', 'LVQRSLL'}),
    'TAb':   ('B', ['164934002'], {'NT_', 'TAB_', 'LOWT'}, {'TWC'}),
    'TInv':  ('B', ['59931005'], {'INVT'}, {'TWO'}),
    'QAb':   ('B', ['164917005'], {'QWAVE'}, {'AQW'}),
    'LPR':   ('B', ['164947007'], {'LPR'}, {'PRIE'}),
    'LAD':   ('B', ['39732003'], {'axis:LAD', 'axis:ALAD'}, {'ALS'}),
    'RAD':   ('B', ['47665007'], {'axis:RAD', 'axis:ARAD'}, {'ARS'}),
}
if not INCLUDE_TIER_B:
    CONCEPTS = {k: v for k, v in CONCEPTS.items() if v[0] == 'A'}
ALL_CONCEPTS = list(CONCEPTS)

# ---- PTB-XL ----
ptb_ok = ptb_meta[ptb_meta.ok].reset_index(drop=True)
def ptb_codes(row):
    d = ast.literal_eval(row.scp_codes)
    c = {k for k, v in d.items() if v >= PTB_MIN_LIKELIHOOD}
    if isinstance(row.heart_axis, str): c.add('axis:' + row.heart_axis)
    return c
ptb_sets = [ptb_codes(r) for r in ptb_ok.itertuples()]
Y_ptb = np.array([[bool(s & CONCEPTS[c][2]) for c in ALL_CONCEPTS] for s in ptb_sets], dtype=np.float32)

# ---- Chapman ----
ch_ok = ch_meta[ch_meta.ok].reset_index(drop=True)
if CH_FORMAT == 'figshare':
    ch_sets = [{str(r.rhythm).strip()} | set(str(r.beat).split()) - {'NONE', 'nan', ''} for r in ch_ok.itertuples()]
else:  # WFDB: SNOMED codes -> acronym via the dataset's own ConditionNames_SNOMED-CT.csv when available
    cn = None
    for root in [Path(DATA_DIR), Path(os.environ.get('CHAPMAN_DIR', '/content/data/chapman'))]:
        hits = list(root.rglob('ConditionNames_SNOMED-CT.csv')) if root.exists() else []
        if hits: cn = pd.read_csv(hits[0]); break
    if cn is not None:
        acr_col = [c for c in cn.columns if 'acronym' in c.lower()][0]
        sn_col = [c for c in cn.columns if 'snomed' in c.lower()][0]
        sn2acr = {str(s).strip(): str(a).strip() for s, a in zip(cn[sn_col], cn[acr_col])}
        print('SNOMED->acronym via', hits[0])
    else:  # fallback: use the SNOMED codes in CONCEPTS (map straight to the concept name as pseudo-acronym)
        print('!!! ConditionNames_SNOMED-CT.csv not found; mapping SNOMED codes via CONCEPTS table (copy the file '
              'into DATA_DIR to use the dataset-provided mapping)')
        sn2acr = {}
        for c, (_, codes, _, acrs) in CONCEPTS.items():
            for code in codes: sn2acr[code] = sorted(acrs)[0]
    ch_sets = [{sn2acr.get(code, 'SNOMED:' + code) for code in str(s).split(',') if code and code != 'nan'}
               for s in ch_ok.snomed]
Y_ch = np.array([[bool(s & CONCEPTS[c][3]) for c in ALL_CONCEPTS] for s in ch_sets], dtype=np.float32)

# unmapped acronyms / codes (transparency)
mapped_ch = set().union(*[v[3] for v in CONCEPTS.values()])
unm = pd.Series([a for s in ch_sets for a in s if a not in mapped_ch]).value_counts()
mapped_ptb = set().union(*[v[2] for v in CONCEPTS.values()])
unm_ptb = pd.Series([a for s in ptb_sets for a in s if a not in mapped_ptb and not a.startswith('axis:')]).value_counts()
print('Chapman labels NOT mapped to any concept (top 25):\n', unm.head(25).to_string())
print('PTB-XL statements NOT mapped (top 15):\n', unm_ptb.head(15).to_string())
unm.to_csv(TAB / 'chapman_unmapped_labels.csv'); unm_ptb.to_csv(TAB / 'ptbxl_unmapped_labels.csv')

In [ ]:
# ---- per-label counts in BOTH datasets, then label selection ----
f = ptb_ok.strat_fold.values
cnt = pd.DataFrame({
    'tier': [CONCEPTS[c][0] for c in ALL_CONCEPTS],
    'ptb_train(1-8)': Y_ptb[f <= 8].sum(0).astype(int), 'ptb_val(9)': Y_ptb[f == 9].sum(0).astype(int),
    'ptb_test(10)': Y_ptb[f == 10].sum(0).astype(int), 'ptb_total': Y_ptb.sum(0).astype(int),
    'chapman_total': Y_ch.sum(0).astype(int)}, index=ALL_CONCEPTS)
cnt['ptb_prev_%'] = (100 * Y_ptb.mean(0)).round(2)
cnt['chapman_prev_%'] = (100 * Y_ch.mean(0)).round(2)
cnt['selected'] = ((cnt['ptb_train(1-8)'] >= MIN_POS['ptb_train']) & (cnt['ptb_test(10)'] >= MIN_POS['ptb_test'])
                   & (cnt['chapman_total'] >= MIN_POS['chapman']))
print(f'N records: PTB-XL {len(Y_ptb)} | Chapman {len(Y_ch)} | rule: {MIN_POS}')
print(cnt.to_string())
cnt.to_csv(TAB / 'label_counts_both_datasets.csv')
LABELS = [c for c in ALL_CONCEPTS if cnt.loc[c, 'selected']]
TIER = {c: CONCEPTS[c][0] for c in LABELS}
C = len(LABELS)
assert C >= 2, 'fewer than 2 labels pass the selection rule'
print(f'\nSELECTED {C} labels:', LABELS)
yP, yC = Y_ptb[:, [ALL_CONCEPTS.index(c) for c in LABELS]], Y_ch[:, [ALL_CONCEPTS.index(c) for c in LABELS]]
co = pd.Series({'ptb': float((yP.sum(1) > 1).mean()), 'chapman': float((yC.sum(1) > 1).mean())})
print('fraction of records with >1 selected label:', co.round(3).to_dict(),
      '-> labels co-occur, so the task is MULTI-LABEL (sigmoid + BCE), not multi-class.')
print('records with 0 selected labels: PTB', int((yP.sum(1) == 0).sum()), '| Chapman', int((yC.sum(1) == 0).sum()),
      '(kept: they are valid all-negative examples)')

fig, ax = plt.subplots(figsize=(max(8, C * 0.6), 4.5))
xx = np.arange(C)
ax.bar(xx - 0.2, cnt.loc[LABELS, 'ptb_prev_%'], 0.4, label='PTB-XL')
ax.bar(xx + 0.2, cnt.loc[LABELS, 'chapman_prev_%'], 0.4, label='Chapman')
ax.set_xticks(xx); ax.set_xticklabels([f'{c}{"*" if TIER[c] == "B" else ""}' for c in LABELS], rotation=60)
ax.set_yscale('log'); ax.set_ylabel('Prevalence (%) [log]'); ax.set_xlabel('Label (* = Tier B, loose definition match)')
ax.set_title('Label prevalence shift between datasets'); ax.legend()
savefig(fig, 'fig01_label_prevalence')

## 3. Signal harmonisation + domain-shift diagnostics
Converter already fixed: lead names/order, mV units, 100 Hz via the same polyphase resampler, 10 s length.
Here: identical band-pass for both, then per-lead standardisation with **training-set** statistics only.

In [ ]:
SOS = butter(FILTER_ORDER, BANDPASS, btype='band', fs=FS, output='sos')

def load_variant(Xraw, meta_ok_idx, variant, max_n=None, rng=None):
    idx = meta_ok_idx if max_n is None or len(meta_ok_idx) <= max_n else np.sort(rng.choice(meta_ok_idx, max_n, replace=False))
    out = np.empty((len(idx), REC_LEN, 12), dtype=np.float32)
    for s in range(0, len(idx), 2000):
        chunk = np.asarray(Xraw[idx[s:s + 2000]], dtype=np.float32)
        out[s:s + 2000] = sosfiltfilt(SOS, chunk, axis=1).astype(np.float32) if variant == 'bp' else chunk
    return out, idx

rng0 = np.random.default_rng(2026)
ptb_rows = np.arange(len(ptb_ok)); ch_rows = np.arange(len(ch_ok))
if FAST_MODE:   # subsample for smoke tests (keeps folds 9/10 complete)
    keep = np.where(f >= 9)[0]
    tr = np.where(f <= 8)[0]; tr = rng0.choice(tr, min(len(tr), FAST_MAX_TRAIN), replace=False)
    ptb_rows = np.sort(np.r_[tr, keep])
    ch_rows = np.sort(rng0.choice(ch_rows, min(len(ch_rows), FAST_MAX_EXT), replace=False))
ptb_X_idx = ptb_ok.idx.values[ptb_rows]; ch_X_idx = ch_ok.idx.values[ch_rows]
yP, yC = yP[ptb_rows], yC[ch_rows]
fold = f[ptb_rows]
meta_P, meta_C = ptb_ok.iloc[ptb_rows].reset_index(drop=True), ch_ok.iloc[ch_rows].reset_index(drop=True)

DATA = {}
def get_data(variant):
    if variant not in DATA:
        DATA.clear(); import gc; gc.collect()   # keep only one preprocessing variant in RAM
        t0 = time.time()
        XP, _ = load_variant(X_ptb_raw, ptb_X_idx, variant)
        XC, _ = load_variant(X_ch_raw, ch_X_idx, variant)
        DATA[variant] = (XP, XC)
        print(f'loaded variant={variant}: PTB {XP.shape}, Chapman {XC.shape} in {time.time() - t0:.0f}s')
    return DATA[variant]
XP, XC = get_data('bp')

In [ ]:
# ---- diagnostics: demographics, amplitude, spectra, domain separability ----
diag = {}
def demo(m):
    a = pd.to_numeric(m.age, errors='coerce'); a = a[(a >= 0) & (a < 120)]
    return {'n': int(len(m)), 'age_mean': float(a.mean()), 'age_sd': float(a.std()),
            'female_%': float(100 * (m.sex == 'F').mean()), 'age<18_%': float(100 * (a < 18).mean())}
diag['demographics'] = {'ptbxl': demo(meta_P), 'chapman': demo(meta_C)}
print(json.dumps(diag['demographics'], indent=1))

def lead_psd(X, n=1500, raw=False, rng=np.random.default_rng(0)):
    sel = rng.choice(len(X), min(n, len(X)), replace=False)
    fr, P = welch(np.asarray(X[sel], dtype=np.float32), fs=FS, nperseg=256, axis=1)
    return fr, np.median(10 * np.log10(P + 1e-12), axis=0)  # (freq, lead)

XPr, _ = load_variant(X_ptb_raw, ptb_X_idx, 'raw', max_n=1500, rng=rng0)
XCr, _ = load_variant(X_ch_raw, ch_X_idx, 'raw', max_n=1500, rng=rng0)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (a, b, ttl) in zip(axs, [(XPr, XCr, 'before band-pass (converter output)'), (XP, XC, f'after {BANDPASS} Hz band-pass')]):
    for X_, lab in [(a, 'PTB-XL'), (b, 'Chapman')]:
        fr, P = lead_psd(X_); ax.plot(fr, P[:, 1], label=lab)
    ax.set_xlabel('Frequency (Hz)'); ax.set_title(f'Lead II median PSD, {ttl}'); ax.legend()
axs[0].set_ylabel('Power (dB, mV²/Hz)')
savefig(fig, 'fig02_psd_lead_II')

fig, ax = plt.subplots(figsize=(7, 4.2))
for m, lab in [(meta_P, 'PTB-XL'), (meta_C, 'Chapman')]:
    ax.hist(np.clip(m.p2p_II, 0, 6), bins=60, alpha=0.55, density=True, label=lab)
ax.set_xlabel('Lead II peak-to-peak amplitude (mV, clipped at 6)'); ax.set_ylabel('Density')
ax.set_title('Amplitude distribution after unit harmonisation'); ax.legend()
savefig(fig, 'fig03_amplitude_lead_II')

# domain classifier: if spectra + amplitudes alone separate the datasets, there is acquisition shift
def psd_feats(X):
    fr, P = welch(np.asarray(X, dtype=np.float32), fs=FS, nperseg=128, axis=1)
    return np.log10(P[:, fr <= 45, :] + 1e-12).reshape(len(X), -1)
n_dc = min(len(XP), len(XC), 1500)
Fd = np.r_[psd_feats(XP[rng0.choice(len(XP), n_dc, replace=False)]), psd_feats(XC[rng0.choice(len(XC), n_dc, replace=False)])]
yd = np.r_[np.zeros(n_dc), np.ones(n_dc)]
pd_ = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1)), Fd, yd,
                        cv=StratifiedKFold(5, shuffle=True, random_state=0), method='predict_proba')[:, 1]
diag['domain_classifier_auroc_psd'] = float(roc_auc_score(yd, pd_))
print(f"Domain classifier (log-PSD features, 5-fold CV) AUROC = {diag['domain_classifier_auroc_psd']:.3f} "
      '(0.5 = indistinguishable; ~1.0 = datasets trivially separable from signal statistics alone)')
del XPr, XCr, XP, XC

## 4. Models
`xresnet1d101` and `inception1d` follow the 1D adaptations benchmarked by Strodthoff et al. 2021
(kernel size 5 xresnet; InceptionTime with kernels 39/19/9). `cnn1d` and `cnn_bilstm` are simple baselines.

In [ ]:
def conv_bn(ni, nf, ks=5, stride=1, act=True, zero_bn=False):
    bn = nn.BatchNorm1d(nf); nn.init.constant_(bn.weight, 0. if zero_bn else 1.)
    layers = [nn.Conv1d(ni, nf, ks, stride, padding=ks // 2, bias=False), bn]
    if act: layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

class XResBlock(nn.Module):
    def __init__(self, expansion, ni, nh, stride=1, ks=5):
        super().__init__()
        nf, ni = nh * expansion, ni * expansion
        if expansion == 1:
            self.convs = nn.Sequential(conv_bn(ni, nh, ks, stride), conv_bn(nh, nf, ks, act=False, zero_bn=True))
        else:
            self.convs = nn.Sequential(conv_bn(ni, nh, 1), conv_bn(nh, nh, ks, stride),
                                       conv_bn(nh, nf, 1, act=False, zero_bn=True))
        idp = []
        if stride != 1: idp.append(nn.AvgPool1d(2, ceil_mode=True))
        if ni != nf: idp.append(conv_bn(ni, nf, 1, act=False))
        self.idpath = nn.Sequential(*idp)
    def forward(self, x):
        a, b = self.convs(x), self.idpath(x)
        return F.relu(a + b[..., :a.shape[-1]])

class ConcatPoolHead(nn.Module):
    def __init__(self, nf, n_out, p=0.5):
        super().__init__()
        self.head = nn.Sequential(nn.BatchNorm1d(2 * nf), nn.Dropout(p), nn.Linear(2 * nf, n_out))
    def forward(self, x):
        return self.head(torch.cat([x.mean(-1), x.amax(-1)], 1))

class XResNet1d(nn.Module):
    def __init__(self, layers, n_out, expansion=4, ks=5, c_in=12):
        super().__init__()
        stem = [conv_bn(c_in, 32, ks, 2), conv_bn(32, 32, ks), conv_bn(32, 64, ks)]
        szs = [64 // expansion, 64, 128, 256, 512]
        blocks = []
        for i, n in enumerate(layers):
            for j in range(n):
                blocks.append(XResBlock(expansion, szs[i] if j == 0 else szs[i + 1], szs[i + 1],
                                        stride=1 if (j > 0 or i == 0) else 2, ks=ks))
        self.body = nn.Sequential(*stem, nn.MaxPool1d(3, 2, 1), *blocks)
        self.head = ConcatPoolHead(512 * expansion, n_out)
    def forward(self, x): return self.head(self.body(x))

class InceptionModule(nn.Module):
    def __init__(self, ni, nf=32, ks=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(ni, bottleneck, 1, bias=False) if ni > 1 else nn.Identity()
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck, nf, k, padding=k // 2, bias=False) for k in ks])
        self.mp = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(ni, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))
    def forward(self, x):
        b = self.bottleneck(x)
        return F.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.mp(x)], 1)))

class Inception1d(nn.Module):
    def __init__(self, n_out, depth=6, nf=32, c_in=12):
        super().__init__()
        self.mods = nn.ModuleList(); self.sc = nn.ModuleList()
        ni = c_in; res_in = c_in
        for d in range(depth):
            self.mods.append(InceptionModule(ni, nf)); ni = nf * 4
            if d % 3 == 2:
                self.sc.append(nn.Sequential(nn.Conv1d(res_in, ni, 1, bias=False), nn.BatchNorm1d(ni))); res_in = ni
        self.head = ConcatPoolHead(ni, n_out)
    def forward(self, x):
        res = x
        for d, m in enumerate(self.mods):
            x = m(x)
            if d % 3 == 2:
                x = F.relu(x + self.sc[d // 3](res)); res = x
        return self.head(x)

class CNN1d(nn.Module):
    def __init__(self, n_out, c_in=12, ch=(64, 128, 128, 256)):
        super().__init__()
        L, ni = [], c_in
        for c in ch: L += [nn.Conv1d(ni, c, 7, padding=3, bias=False), nn.BatchNorm1d(c), nn.ReLU(inplace=True), nn.MaxPool1d(2)]; ni = c
        self.body = nn.Sequential(*L); self.head = ConcatPoolHead(ni, n_out)
    def forward(self, x): return self.head(self.body(x))

class CNNBiLSTM(nn.Module):
    def __init__(self, n_out, c_in=12, hid=128):
        super().__init__()
        L, ni = [], c_in
        for c in (64, 128, 128): L += [nn.Conv1d(ni, c, 7, padding=3, bias=False), nn.BatchNorm1d(c), nn.ReLU(inplace=True), nn.MaxPool1d(2)]; ni = c
        self.cnn = nn.Sequential(*L)
        self.lstm = nn.LSTM(ni, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = ConcatPoolHead(2 * hid, n_out)
    def forward(self, x):
        h, _ = self.lstm(self.cnn(x).transpose(1, 2))
        return self.head(h.transpose(1, 2))

def build_model(name, n_out):
    return {'xresnet1d101': lambda: XResNet1d([3, 4, 23, 3], n_out), 'xresnet1d50': lambda: XResNet1d([3, 4, 6, 3], n_out),
            'inception1d': lambda: Inception1d(n_out), 'cnn1d': lambda: CNN1d(n_out),
            'cnn_bilstm': lambda: CNNBiLSTM(n_out)}[name]()

for m in sorted(set(MODELS + REVERSE_MODELS)):
    net = build_model(m, C); out_ = net(torch.randn(2, 12, WIN))
    print(f'{m:14s} params={sum(p.numel() for p in net.parameters()) / 1e6:5.2f}M  out={tuple(out_.shape)}')

## 5. Training / inference (resumable)

In [ ]:
def make_splits(direction):
    """returns dict of (X, y, meta) for train / val / test_in / test_ext. External set is never touched in training."""
    if direction == 'ptb2chap':
        tr, va, te = np.where(fold <= 8)[0], np.where(fold == 9)[0], np.where(fold == 10)[0]
        return dict(src='P', tr=tr, va=va, te=te, ext='C')
    r = np.random.default_rng(2026).permutation(len(yC))
    n_tr, n_va = int(0.8 * len(r)), int(0.1 * len(r))
    return dict(src='C', tr=np.sort(r[:n_tr]), va=np.sort(r[n_tr:n_tr + n_va]), te=np.sort(r[n_tr + n_va:]), ext='P')

def arrays(variant, key):
    XP_, XC_ = get_data(variant)
    return (XP_, yP) if key == 'P' else (XC_, yC)

def lead_stats(X, idx, n=4000, rng=np.random.default_rng(0)):
    s = X[np.sort(rng.choice(idx, min(n, len(idx)), replace=False))]
    return s.mean((0, 1)).astype(np.float32), (s.std((0, 1)) + 1e-6).astype(np.float32)

@torch.no_grad()
def predict(model, X, idx, mu, sd, bs=256):
    model.eval(); out = []
    starts = np.arange(0, REC_LEN - WIN + 1, TEST_STRIDE)
    mu_t, sd_t = torch.tensor(mu, device=DEV), torch.tensor(sd, device=DEV)
    for s in range(0, len(idx), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[idx[s:s + bs]])).to(DEV)
        xb = ((xb - mu_t) / sd_t).transpose(1, 2)                        # (B,12,1000)
        w = torch.stack([xb[..., k:k + WIN] for k in starts], 1)         # (B,nw,12,WIN)
        with torch.autocast(DEV.type, enabled=DEV.type == 'cuda'):
            p = torch.sigmoid(model(w.flatten(0, 1)).float())
        out.append(p.view(len(xb), len(starts), -1).amax(1).cpu().numpy())  # element-wise max over windows
    return np.concatenate(out)

def train_run(run_id, direction, model_name, seed, variant):
    pred_f, ck_f = OUT / 'preds' / f'{run_id}.npz', OUT / 'checkpoints' / f'{run_id}.pt'
    if pred_f.exists():
        print(f'[skip] {run_id} already finished'); return
    sp = make_splits(direction)
    Xs, ys = arrays(variant, sp['src']); Xe, ye = arrays(variant, sp['ext'])
    mu, sd = lead_stats(Xs, sp['tr'])
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    model = build_model(model_name, C).to(DEV)
    pos = ys[sp['tr']].sum(0); neg = len(sp['tr']) - pos
    pos_w = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, POS_WEIGHT_CLIP), dtype=torch.float32, device=DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    steps = math.ceil(len(sp['tr']) / BATCH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * steps)
    scaler = torch.amp.GradScaler('cuda', enabled=DEV.type == 'cuda')
    st = dict(epoch=0, best=-1, best_state=None, hist=[], bad=0)
    if ck_f.exists():
        ck = torch.load(ck_f, map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler']); st = ck['st']; rng = ck['rng']
        print(f'[resume] {run_id} from epoch {st["epoch"]}')
    mu_t, sd_t = torch.tensor(mu, device=DEV), torch.tensor(sd, device=DEV)
    ar = np.arange(WIN)
    while st['epoch'] < EPOCHS and st['bad'] < PATIENCE:
        model.train(); t0 = time.time(); perm = rng.permutation(sp['tr']); tl = 0.
        for b in range(steps):
            bi = perm[b * BATCH:(b + 1) * BATCH]
            off = rng.integers(0, REC_LEN - WIN + 1, len(bi))
            xb = torch.from_numpy(Xs[bi[:, None], off[:, None] + ar]).to(DEV)   # random 2.5 s crop
            xb = ((xb - mu_t) / sd_t).transpose(1, 2)
            yb = torch.from_numpy(ys[bi]).to(DEV)
            with torch.autocast(DEV.type, enabled=DEV.type == 'cuda'):
                loss = crit(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item()
        pv = predict(model, Xs, sp['va'], mu, sd)
        aucs = [roc_auc_score(ys[sp['va'], c], pv[:, c]) for c in range(C) if 0 < ys[sp['va'], c].sum() < len(sp['va'])]
        vauc = float(np.mean(aucs)); st['epoch'] += 1
        st['hist'].append({'epoch': st['epoch'], 'train_loss': tl / steps, 'val_macro_auroc': vauc, 'sec': time.time() - t0})
        if vauc > st['best']:
            st['best'], st['bad'] = vauc, 0
            st['best_state'] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st['bad'] += 1
        print(f'  {run_id} ep{st["epoch"]:3d} loss {tl / steps:.4f} val macroAUROC {vauc:.4f} '
              f'best {st["best"]:.4f} ({time.time() - t0:.0f}s)')
        tmp = ck_f.with_suffix('.tmp')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'scaler': scaler.state_dict(), 'st': st, 'rng': rng}, tmp); tmp.replace(ck_f)
    model.load_state_dict(st['best_state'])
    torch.save({'model': st['best_state'], 'labels': LABELS, 'mu': mu, 'sd': sd, 'model_name': model_name,
                'bandpass': BANDPASS if variant == 'bp' else None}, OUT / 'checkpoints' / f'{run_id}_best.pt')
    ext_idx = np.arange(len(ye))
    np.savez_compressed(pred_f, val_p=predict(model, Xs, sp['va'], mu, sd), val_y=ys[sp['va']],
                        in_p=predict(model, Xs, sp['te'], mu, sd), in_y=ys[sp['te']], in_idx=sp['te'],
                        ext_p=predict(model, Xe, ext_idx, mu, sd), ext_y=ye, hist=json.dumps(st['hist']))
    ck_f.unlink(missing_ok=True)   # keep only the best weights once finished
    print(f'[done] {run_id}')

In [ ]:
RUNS = [('ptb2chap', m, s, 'bp') for m in MODELS for s in SEEDS]
RUNS += [('chap2ptb', m, s, 'bp') for m in REVERSE_MODELS for s in SEEDS]
if ABLATION_NOFILTER_MODEL:
    RUNS += [('ptb2chap', ABLATION_NOFILTER_MODEL, s, 'raw') for s in SEEDS]
run_id = lambda d, m, s, v: f'{d}__{m}__{v}__s{s}'
print(len(RUNS), 'runs planned;', sum((OUT / 'preds' / f'{run_id(*r)}.npz').exists() for r in RUNS), 'already done')
T_ALL = time.time()
for r in RUNS:
    train_run(run_id(*r), *r)
print(f'all runs finished in {(time.time() - T_ALL) / 60:.1f} min this session')

## 6. Metrics: point estimates (per seed), mean ± std over seeds, bootstrap 95 % CIs
Bootstrap: B resamples of each test set (same resamples for every model ⇒ paired model comparisons).
For a resample, the metric is averaged over seeds; CI = 2.5/97.5 percentiles. Resamples where a class has no
positives are discarded for macro metrics (as in Strodthoff et al. 2021). Drop CI uses independent resamples
of the in-domain and external sets.

In [ ]:
def boot_weights(n, B, seed):
    rng = np.random.default_rng(seed)
    return np.stack([np.bincount(rng.integers(0, n, n), minlength=n) for _ in range(B)]).astype(np.float32)

def wauc(y, s, W, chunk=100):
    """weighted (bootstrap-count) AUROC, exact incl. ties; y (n,), s (n,), W (B,n) -> (B,)"""
    o = np.argsort(s, kind='mergesort'); ss, yy = s[o], y[o].astype(bool)
    st = np.r_[0, np.flatnonzero(np.diff(ss)) + 1]
    res = []
    for b in range(0, len(W), chunk):
        Wb = W[b:b + chunk][:, o]
        Wp = np.add.reduceat(Wb * yy, st, axis=1); Wn = np.add.reduceat(Wb * ~yy, st, axis=1)
        num = (Wp * (np.cumsum(Wn, 1) - Wn + 0.5 * Wn)).sum(1); den = Wp.sum(1) * Wn.sum(1)
        res.append(np.where(den > 0, num / np.maximum(den, 1e-12), np.nan))
    return np.concatenate(res)

def auc_matrix(y, p, W):   # (B, C)
    return np.stack([wauc(y[:, c], p[:, c], W) for c in range(y.shape[1])], 1)

def conf_counts(y, pred, W):
    yb, pb = y.astype(np.float32), pred.astype(np.float32)
    return W @ (yb * pb), W @ ((1 - yb) * pb), W @ (yb * (1 - pb)), W @ ((1 - yb) * (1 - pb))

def f1_sens_spec(y, pred, W):
    tp, fp, fn, tn = conf_counts(y, pred, W)
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(tp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
        return f1, np.where(tp + fn > 0, tp / (tp + fn), np.nan), np.where(tn + fp > 0, tn / (tn + fp), np.nan)

def best_thresholds(y, p):
    th = np.full(y.shape[1], 0.5)
    for c in range(y.shape[1]):
        if 0 < y[:, c].sum() < len(y):
            pr, rc, t = precision_recall_curve(y[:, c], p[:, c])
            f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-12)
            th[c] = t[np.argmax(f1[:-1])]
    return th

def point_auc(y, p):
    return np.array([roc_auc_score(y[:, c], p[:, c]) if 0 < y[:, c].sum() < len(y) else np.nan for c in range(y.shape[1])])

def ci(a):
    a = np.asarray(a); a = a[~np.isnan(a)]
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))] if len(a) else [np.nan, np.nan]

def macro_rows(M):  # drop bootstrap rows with any NaN class
    ok = ~np.isnan(M).any(1); return M[ok].mean(1)

def evaluate_group(runs_preds, Ws):
    """runs_preds: list over seeds of npz dicts. Ws: {'in': W, 'ext': W}"""
    R = {'n_seeds': len(runs_preds)}
    seeds = []
    for P in runs_preds:
        th = best_thresholds(P['val_y'], P['val_p']); one = {'thresholds': th.tolist()}
        for split in ['in', 'ext']:
            y, p = P[f'{split}_y'], P[f'{split}_p']; pred = p >= th
            f1, se, sp_ = [v[0] for v in f1_sens_spec(y, pred, np.ones((1, len(y)), np.float32))]
            one[split] = dict(auc=point_auc(y, p), f1=f1, sens=se, spec=sp_)
            if split == 'ext':   # ORACLE: thresholds re-tuned on the external set itself (analysis only, not deployable)
                pr_o = p >= best_thresholds(y, p)
                one['ext_oracle_f1'] = f1_sens_spec(y, pr_o, np.ones((1, len(y)), np.float32))[0][0]
        seeds.append(one)
    for split in ['in', 'ext']:
        A = np.stack([s[split]['auc'] for s in seeds]); F1 = np.stack([s[split]['f1'] for s in seeds])
        SE = np.stack([s[split]['sens'] for s in seeds]); SP = np.stack([s[split]['spec'] for s in seeds])
        Ab = np.mean([auc_matrix(P[f'{split}_y'], P[f'{split}_p'], Ws[split]) for P in runs_preds], 0)
        fb = [f1_sens_spec(P[f'{split}_y'], P[f'{split}_p'] >= np.array(s['thresholds']), Ws[split])
              for P, s in zip(runs_preds, seeds)]
        F1b, SEb, SPb = [np.mean([x[k] for x in fb], 0) for k in range(3)]
        R[split] = {
            'macro_auroc_mean': float(np.nanmean(A, 1).mean()), 'macro_auroc_std': float(np.nanmean(A, 1).std()),
            'macro_auroc_ci95': ci(macro_rows(Ab)),
            'macro_f1_mean': float(np.nanmean(F1, 1).mean()), 'macro_f1_std': float(np.nanmean(F1, 1).std()),
            'macro_f1_ci95': ci(macro_rows(F1b)),
            'per_class': {LABELS[c]: {
                'auroc_mean': float(A[:, c].mean()), 'auroc_std': float(A[:, c].std()), 'auroc_ci95': ci(Ab[:, c]),
                'f1_mean': float(F1[:, c].mean()), 'f1_ci95': ci(F1b[:, c]),
                'sens_mean': float(SE[:, c].mean()), 'sens_ci95': ci(SEb[:, c]),
                'spec_mean': float(SP[:, c].mean()), 'spec_ci95': ci(SPb[:, c]),
                'n_pos': int(runs_preds[0][f'{split}_y'][:, c].sum())} for c in range(C)},
            '_boot_auc': Ab, '_boot_f1': F1b}
    R['ext_oracle_macro_f1_mean'] = float(np.mean([np.nanmean(s['ext_oracle_f1']) for s in seeds]))
    d = macro_rows(R['in']['_boot_auc'])[:min(len(macro_rows(R['in']['_boot_auc'])), len(macro_rows(R['ext']['_boot_auc'])))]
    e = macro_rows(R['ext']['_boot_auc'])[:len(d)]
    R['auroc_drop'] = {'mean': R['in']['macro_auroc_mean'] - R['ext']['macro_auroc_mean'], 'ci95': ci(d - e)}
    R['per_class_auroc_drop'] = {LABELS[c]: {'mean': R['in']['per_class'][LABELS[c]]['auroc_mean'] - R['ext']['per_class'][LABELS[c]]['auroc_mean'],
                                             'ci95': ci(R['in']['_boot_auc'][:, c] - R['ext']['_boot_auc'][:, c])} for c in range(C)}
    for t in ['A', 'B']:
        cs = [c for c in range(C) if TIER[LABELS[c]] == t]
        if cs:
            dd = R['in']['_boot_auc'][:, cs] - R['ext']['_boot_auc'][:, cs]
            R[f'tier{t}_auroc_drop'] = {'mean': float(np.mean([R['per_class_auroc_drop'][LABELS[c]]['mean'] for c in cs])),
                                        'ci95': ci(macro_rows(dd)), 'labels': [LABELS[c] for c in cs]}
    # calibration-in-the-large: mean predicted prob vs observed prevalence (seed-averaged predictions)
    R['calibration'] = {split: {LABELS[c]: {'mean_pred': float(np.mean([P[f'{split}_p'][:, c].mean() for P in runs_preds])),
                                            'prevalence': float(runs_preds[0][f'{split}_y'][:, c].mean())} for c in range(C)}
                        for split in ['in', 'ext']}
    return R

def load_preds(d, m, v):
    out = []
    for s in SEEDS:
        f_ = OUT / 'preds' / f'{run_id(d, m, s, v)}.npz'
        if f_.exists():
            z = np.load(f_); out.append({k: z[k] for k in z.files})
    return out

RESULTS = {}
groups = sorted({(d, m, v) for d, m, s, v in RUNS})
for d in sorted({g[0] for g in groups}):
    any_p = next(p for g in groups if g[0] == d for p in [load_preds(*g)] if p)
    Ws = {'in': boot_weights(len(any_p[0]['in_y']), N_BOOT, 11), 'ext': boot_weights(len(any_p[0]['ext_y']), N_BOOT, 12)}
    for g in [g for g in groups if g[0] == d]:
        P = load_preds(*g)
        if not P: continue
        t0 = time.time(); RESULTS['|'.join(g)] = evaluate_group(P, Ws)
        r = RESULTS['|'.join(g)]
        print(f"{'|'.join(g):34s} seeds={len(P)}  in AUROC {r['in']['macro_auroc_mean']:.3f}±{r['in']['macro_auroc_std']:.3f} "
              f"ext {r['ext']['macro_auroc_mean']:.3f}±{r['ext']['macro_auroc_std']:.3f}  "
              f"drop {r['auroc_drop']['mean']:.3f} {np.round(r['auroc_drop']['ci95'], 3).tolist()}  ({time.time() - t0:.0f}s)")
    # paired comparison vs the best external model in this direction (same bootstrap resamples)
    keys = [k for k in RESULTS if k.startswith(d + '|') and k.endswith('|bp')]
    if keys:
        best = max(keys, key=lambda k: RESULTS[k]['ext']['macro_auroc_mean'])
        for k in keys:
            a, b = macro_rows(RESULTS[best]['ext']['_boot_auc']), macro_rows(RESULTS[k]['ext']['_boot_auc'])
            n = min(len(a), len(b))
            RESULTS[k]['ext_delta_vs_best'] = {'best': best, 'mean': float(a[:n].mean() - b[:n].mean()), 'ci95': ci(a[:n] - b[:n])}

## 7. Tables + figures

In [ ]:
rows = []
for k, r in RESULTS.items():
    d, m, v = k.split('|')
    fmt = lambda x, s, c: f'{x:.3f}±{s:.3f} [{c[0]:.3f},{c[1]:.3f}]'
    rows.append({'direction': d, 'model': m, 'variant': v, 'seeds': r['n_seeds'],
                 'in_macroAUROC': fmt(r['in']['macro_auroc_mean'], r['in']['macro_auroc_std'], r['in']['macro_auroc_ci95']),
                 'ext_macroAUROC': fmt(r['ext']['macro_auroc_mean'], r['ext']['macro_auroc_std'], r['ext']['macro_auroc_ci95']),
                 'AUROC_drop [CI]': f"{r['auroc_drop']['mean']:.3f} [{r['auroc_drop']['ci95'][0]:.3f},{r['auroc_drop']['ci95'][1]:.3f}]",
                 'in_macroF1': fmt(r['in']['macro_f1_mean'], r['in']['macro_f1_std'], r['in']['macro_f1_ci95']),
                 'ext_macroF1(val thr)': fmt(r['ext']['macro_f1_mean'], r['ext']['macro_f1_std'], r['ext']['macro_f1_ci95']),
                 'ext_macroF1(ORACLE thr)': f"{r['ext_oracle_macro_f1_mean']:.3f}"})
main_tab = pd.DataFrame(rows); main_tab.to_csv(TAB / 'main_results.csv', index=False)
print(BANNER); print(main_tab.to_string(index=False))

pc = []
for k, r in RESULTS.items():
    for c in LABELS:
        pc.append({'group': k, 'label': c, 'tier': TIER[c], **{f'{s}_{q}': r[s]['per_class'][c][q] for s in ['in', 'ext']
                   for q in ['n_pos', 'auroc_mean', 'auroc_std', 'f1_mean', 'sens_mean', 'spec_mean']},
                   'auroc_drop': r['per_class_auroc_drop'][c]['mean'],
                   'drop_ci_lo': r['per_class_auroc_drop'][c]['ci95'][0], 'drop_ci_hi': r['per_class_auroc_drop'][c]['ci95'][1]})
per_class = pd.DataFrame(pc); per_class.to_csv(TAB / 'per_class_results.csv', index=False)

In [ ]:
for d in sorted({k.split('|')[0] for k in RESULTS}):
    ks = [k for k in RESULTS if k.startswith(d + '|')]
    fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(ks)), 4.5)); x = np.arange(len(ks))
    for off, split, lab in [(-0.2, 'in', 'in-domain test'), (0.2, 'ext', 'external test')]:
        mu_ = [RESULTS[k][split]['macro_auroc_mean'] for k in ks]
        lo = [mu_[i] - RESULTS[k][split]['macro_auroc_ci95'][0] for i, k in enumerate(ks)]
        hi = [RESULTS[k][split]['macro_auroc_ci95'][1] - mu_[i] for i, k in enumerate(ks)]
        ax.bar(x + off, mu_, 0.4, yerr=[lo, hi], capsize=3, label=lab)
    ax.set_xticks(x); ax.set_xticklabels([k.split('|', 1)[1].replace('|', '\n') for k in ks])
    lo_all = min(RESULTS[k][sp_]['macro_auroc_ci95'][0] for k in ks for sp_ in ['in', 'ext'])
    ax.set_ylim(min(0.5, np.floor(lo_all * 20) / 20), 1.0); ax.set_ylabel('Macro AUROC (mean over seeds, 95% CI)'); ax.set_xlabel('Model | preprocessing')
    src_name, ext_name = ('PTB-XL', 'Chapman') if d == 'ptb2chap' else ('Chapman', 'PTB-XL')
    ax.set_title(f'Train {src_name}: in-domain vs external ({ext_name})')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False)
    savefig(fig, f'fig04_macro_auroc_{d}')

    best = max([k for k in ks if k.endswith('|bp')], key=lambda k: RESULTS[k]['ext']['macro_auroc_mean'])
    r = RESULTS[best]; order = sorted(LABELS, key=lambda c: -r['per_class_auroc_drop'][c]['mean'])
    dm = [r['per_class_auroc_drop'][c]['mean'] for c in order]
    err = [[dm[i] - r['per_class_auroc_drop'][c]['ci95'][0] for i, c in enumerate(order)],
           [r['per_class_auroc_drop'][c]['ci95'][1] - dm[i] for i, c in enumerate(order)]]
    fig, ax = plt.subplots(figsize=(max(8, 0.6 * C), 4.5))
    ax.bar(range(C), dm, yerr=err, capsize=3, color=['C3' if TIER[c] == 'B' else 'C0' for c in order])
    ax.axhline(0, color='k', lw=0.8); ax.set_xticks(range(C)); ax.set_xticklabels(order, rotation=60)
    ax.set_ylabel('AUROC drop (in-domain − external)'); ax.set_xlabel('Label (red = Tier B)')
    ax.set_title(f'Per-class AUROC drop, {best.split("|")[1]} ({src_name}→{ext_name})')
    savefig(fig, f'fig05_per_class_drop_{d}')

    # why? prevalence shift vs drop
    sp = make_splits(d); ytr = (yP if sp['src'] == 'P' else yC)[sp['tr']]; yex = yC if sp['ext'] == 'C' else yP
    lpr = np.log2((yex.mean(0) + 1e-4) / (ytr.mean(0) + 1e-4))
    rho, pval = spearmanr(lpr, [r['per_class_auroc_drop'][c]['mean'] for c in LABELS])
    RESULTS[best]['prevalence_shift_vs_drop_spearman'] = {'rho': float(rho), 'p': float(pval), 'n_labels': C}
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(lpr, [r['per_class_auroc_drop'][c]['mean'] for c in LABELS], c=['C3' if TIER[c] == 'B' else 'C0' for c in LABELS])
    for i, c in enumerate(LABELS): ax.annotate(c, (lpr[i], r['per_class_auroc_drop'][c]['mean']), fontsize=9)
    ax.axhline(0, color='k', lw=0.6); ax.set_xlabel('log2(external prevalence / train prevalence)')
    ax.set_ylabel('AUROC drop'); ax.set_title(f'Prevalence shift vs AUROC drop (Spearman ρ={rho:.2f}, n={C})')
    savefig(fig, f'fig06_prevalence_vs_drop_{d}')

    # threshold transfer: F1 with val thresholds vs oracle thresholds; calibration-in-the-large
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    P_ = load_preds(*best.split('|'))
    f1_val = [r['ext']['per_class'][c]['f1_mean'] for c in LABELS]
    f1_or = []
    for c in range(C):
        f1_or.append(np.mean([f1_sens_spec(p_['ext_y'][:, [c]], p_['ext_p'][:, [c]] >= best_thresholds(p_['ext_y'][:, [c]], p_['ext_p'][:, [c]]),
                                           np.ones((1, len(p_['ext_y'])), np.float32))[0][0, 0] for p_ in P_]))
    axs[0].bar(np.arange(C) - 0.2, f1_val, 0.4, label='val-tuned threshold (deployable)')
    axs[0].bar(np.arange(C) + 0.2, f1_or, 0.4, label='oracle threshold (external-tuned)')
    axs[0].set_xticks(range(C)); axs[0].set_xticklabels(LABELS, rotation=60); axs[0].set_ylabel('External F1')
    axs[0].set_xlabel('Label'); axs[0].set_title('Threshold transfer loss on external set'); axs[0].legend(fontsize=9)
    cal = r['calibration']['ext']
    axs[1].scatter([cal[c]['prevalence'] for c in LABELS], [cal[c]['mean_pred'] for c in LABELS])
    for c in LABELS: axs[1].annotate(c, (cal[c]['prevalence'], cal[c]['mean_pred']), fontsize=9)
    lim = [1e-3, 1]; axs[1].plot(lim, lim, 'k--', lw=0.8); axs[1].set_xscale('log'); axs[1].set_yscale('log')
    axs[1].set_xlabel('Observed prevalence (external)'); axs[1].set_ylabel('Mean predicted probability')
    axs[1].set_title('Calibration-in-the-large (external; weighted loss inflates probs)')
    savefig(fig, f'fig07_threshold_calibration_{d}')

# age-stratified external AUROC (forward direction, best model) — sensitivity to age shift
kf = [k for k in RESULTS if k.startswith('ptb2chap|') and k.endswith('|bp')]
if kf:
    best = max(kf, key=lambda k: RESULTS[k]['ext']['macro_auroc_mean']); P_ = load_preds(*best.split('|'))
    age = pd.to_numeric(meta_C.age, errors='coerce').values; bins = [0, 18, 40, 60, 80, 200]; agerows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        msk = (age >= lo) & (age < hi)
        if msk.sum() < 50: continue
        a = np.nanmean([np.nanmean(point_auc(p_['ext_y'][msk], p_['ext_p'][msk])) for p_ in P_])
        agerows.append({'age_bin': f'{lo}-{hi}', 'n': int(msk.sum()), 'ext_macro_auroc': float(a)})
    age_tab = pd.DataFrame(agerows); age_tab.to_csv(TAB / 'external_auroc_by_age.csv', index=False)
    print('External macro-AUROC by age (', best, '):\n', age_tab.to_string(index=False))
    diag['external_auroc_by_age'] = agerows

# learning curves
fig, ax = plt.subplots(figsize=(8, 4.5))
for r_ in RUNS:
    f_ = OUT / 'preds' / f'{run_id(*r_)}.npz'
    if f_.exists() and r_[2] == SEEDS[0]:
        h = pd.DataFrame(json.loads(str(np.load(f_)['hist'])))
        ax.plot(h.epoch, h.val_macro_auroc, label=f'{r_[0]}|{r_[1]}|{r_[3]}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation macro AUROC'); ax.set_title(f'Validation curves (seed {SEEDS[0]})')
ax.legend(fontsize=8); savefig(fig, 'fig08_learning_curves')

## 8. results.json

In [ ]:
def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items() if not str(k).startswith('_')}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.ndarray): return clean(o.tolist())
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    return o

results = {
    'SYNTHETIC_DATA': SYNTHETIC, 'FAST_MODE': FAST_MODE,
    'warning': 'SYNTHETIC/FAST numbers are meaningless' if (SYNTHETIC or FAST_MODE) else 'real full run',
    'config': dict(FS=FS, WIN=WIN, TEST_STRIDE=TEST_STRIDE, BANDPASS=BANDPASS, EPOCHS=EPOCHS, PATIENCE=PATIENCE,
                   BATCH=BATCH, LR=LR, WD=WD, SEEDS=SEEDS, N_BOOT=N_BOOT, MIN_POS=MIN_POS, MODELS=MODELS,
                   REVERSE_MODELS=REVERSE_MODELS, ABLATION=ABLATION_NOFILTER_MODEL, chapman_format=CH_FORMAT,
                   torch=torch.__version__, gpu=torch.cuda.get_device_name(0) if DEV.type == 'cuda' else 'cpu'),
    'conversion_report': conv_report, 'labels': LABELS, 'tiers': TIER,
    'label_counts': cnt.reset_index().rename(columns={'index': 'label'}).to_dict('records'),
    'diagnostics': diag, 'results': RESULTS}
json.dump(clean(results), open(OUT / 'results.json', 'w'), indent=1)
print(BANNER)
print('saved', OUT / 'results.json')
print('figures:', sorted(p.name for p in FIG.glob('*.png')))